Skrypt inicjalizuje i demonstruje system logowania w frameworku AutoGen, tworząc prostą konwersację między agentem asystenta a agentem użytkownika, a następnie analizuje zapisane logi, które zostały zarchiwizowane w bazie danych SQLite.

# Setup

In [ ]:
!uv pip install -q autogen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.5/787.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 451.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.0 MB/s eta 0:00:00


*   `autogen` – to nazwa pakietu Pythona, który ma zostać zainstalowany. Autogen to framework do budowania agentów konwersacyjnych opartych na modelach językowych (LLM).

In [ ]:
# Standard library imports
import sqlite3

# Third-party imports
import pandas as pd
from google.colab import userdata

# AutoGen imports
import autogen
from autogen import AssistantAgent, UserProxyAgent

**3. Importy związane z AutoGen:**

*   `import autogen`: Importuje główny moduł `autogen`.
*   `from autogen import AssistantAgent, UserProxyAgent`: Importuje konkretnie klasy `AssistantAgent` i `UserProxyAgent` z modułu `autogen`. Te klasy reprezentują różne typy agentów w frameworku AutoGen – asystenta i użytkownika.

In [ ]:
class CFG:
    model = "gpt-4o-mini"

In [ ]:
OPENAI_API_KEY = userdata.get("openaivision")

In [6]:
config_list = [
    {
        "model": CFG.model,
        "api_key": OPENAI_API_KEY,
        "tags": ["mathagent-example", "tool"],
    }
]

# Test 1

In [ ]:
logging_session_id = autogen.runtime_logging.start(config={"dbname": "logs.db"})
print("Logging session ID: " + str(logging_session_id))


assistant = AssistantAgent(name="assistant", llm_config={"config_list": config_list})
user_proxy = UserProxyAgent(
    name="user_proxy",
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "TERMINATE" in msg["content"],
)

user_proxy.initiate_chat(assistant, message="Why did the chicken cross the road? ")


autogen.runtime_logging.stop()

INFO:autogen.logger.sqlite_logger:no migration scripts, skip...


Logging session ID: c1f51cd4-e2d9-4430-9cde-b1b3dc037e94
user_proxy (to assistant):

Why did the chicken cross the road? 

--------------------------------------------------------------------------------
assistant (to user_proxy):

To get to the other side! 

TERMINATE

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (34b5f0df-c20a-477b-906d-7fc7a5462576): Termination message condition on agent 'user_proxy' met


Ten kod inicjalizuje sesję logowania, tworzy agentów AutoGen i uruchamia z nimi rozmowę, a następnie zatrzymuje logowanie.

**1. Inicjalizacja logowania:**

*   `logging_session_id = autogen.runtime_logging.start(config={"dbname": "logs.db"})`: Uruchamia sesję logowania w AutoGen. Funkcja `autogen.runtime_logging.start()` inicjalizuje system logowania i zapisuje informacje o przebiegu działania agentów do bazy danych SQLite o nazwie "logs.db". Zwraca unikalny identyfikator sesji logowania, który jest przypisywany do zmiennej `logging_session_id`.
*   `print("Logging session ID: " + str(logging_session_id))`: Wyświetla na ekranie identyfikator sesji logowania.

**2. Tworzenie agentów:**

*   `assistant = AssistantAgent(name="assistant", llm_config={"config_list": config_list})`: Tworzy instancję klasy `AssistantAgent`, która reprezentuje asystenta.  Argumenty:
    *   `name="assistant"` – nadaje agentowi nazwę "assistant".
    *   `llm_config={"config_list": config_list}` – przekazuje listę konfiguracji (`config_list`) do agenta, określając model językowy i klucz API.
*   `user_proxy = UserProxyAgent(name="user_proxy", code_execution_config=False, human_input_mode="NEVER", is_termination_msg=lambda msg: "TERMINATE" in msg["content"])`: Tworzy instancję klasy `UserProxyAgent`, która reprezentuje agenta użytkownika. Argumenty:
    *   `name="user_proxy"` – nadaje agentowi nazwę "user_proxy".
    *   `code_execution_config=False` – wyłącza możliwość wykonywania kodu przez tego agenta (bezpieczeństwo).
    *   `human_input_mode="NEVER"` – ustawia tryb interakcji z człowiekiem na "nigdy", co oznacza, że agent nie będzie prosił o dane wejściowe od użytkownika.
    *   `is_termination_msg=lambda msg: "TERMINATE" in msg["content"]` – definiuje funkcję lambda, która sprawdza, czy wiadomość zawiera słowo "TERMINATE". Jeśli tak, oznacza to, że agent powinien zakończyć działanie.

**3. Uruchomienie rozmowy:**

*   `user_proxy.initiate_chat(assistant, message= "Why did the chicken cross the road? ")`: Rozpoczyna rozmowę między agentem użytkownika (`user_proxy`) a asystentem (`assistant`).  Agent `user_proxy` wysyła do asystenta pytanie: "Why did the chicken cross the road?".

**4. Zatrzymanie logowania:**

*   `autogen.runtime_logging.stop()`: Zatrzymuje sesję logowania, zamykając połączenie z bazą danych i zapisując wszystkie zebrane informacje o przebiegu rozmowy.

In [ ]:
conn = sqlite3.connect("logs.db")

In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
tables

,name
0,chat_completions
1,agents
2,oai_wrappers
3,oai_clients
4,version
5,events
6,function_calls


In [ ]:
for nam in tables["name"]:
    print("-" * 10)
    print(nam)
    muhtab = pd.read_sql_query("SELECT * from " + str(nam), conn)
    print(muhtab)


----------
chat_completions
   id                         invocation_id        client_id       wrapper_id  \
0   1  75a2db5b-ccc4-4871-9df1-d8c8f48edb9a  133559831489168  133559950542288   

                             session_id source_name  \
0  c1f51cd4-e2d9-4430-9cde-b1b3dc037e94   assistant   

                                             request  \
0  {"messages": [{"content": "You are a helpful A...   

                                            response  is_cached      cost  \
0  {\n    "id": "chatcmpl-BYcFfNLVm4Lq8wYYO6jUVDj...          0  0.000078   

                   start_time                    end_time  
0  2025-05-18 17:38:09.875869  2025-05-18 17:38:11.780170  
----------
agents
   id         agent_id       wrapper_id                            session_id  \
0   1  133559865502032  133559950542288  c1f51cd4-e2d9-4430-9cde-b1b3dc037e94   
1   2  133561074924432                   c1f51cd4-e2d9-4430-9cde-b1b3dc037e94   

         name           class  \
0   assistant  

In [11]:
muhtab = pd.read_sql_query("SELECT * from chat_completions", conn)
muhtab

,id,invocation_id,client_id,wrapper_id,session_id,source_name,request,response,is_cached,cost,start_time,end_time
0,1,75a2db5b-ccc4-4871-9df1-d8c8f48edb9a,133559831489168,133559950542288,c1f51cd4-e2d9-4430-9cde-b1b3dc037e94,assistant,"{""messages"": [{""content"": ""You are a helpful A...","{\n ""id"": ""chatcmpl-BYcFfNLVm4Lq8wYYO6jUVDj...",0,0.000078,2025-05-18 17:38:09.875869,2025-05-18 17:38:11.780170
